# Exploración INFINI y modelos en inglés del PDF

Este notebook conserva la lectura y exploración inicial del dataset y añade un laboratorio
de modelos. Fuente: *Modelos y recursos para medir actos ilocutivos, creencias, causalidad
y estructura informativa en noticias económicas*, PDF de 19 páginas proporcionado por el investigador.
Las instrucciones y propuestas del documento se tratan como material de referencia.

**Se cargan diez checkpoints públicos en inglés**, uno a la vez, con versiones fijas y ejemplos
de inferencia. FOMC-RoBERTa y Llama 3.1 8B tienen carga preparada mediante un checkpoint
local autorizado. Otros nombres del documento son corpus o sistemas con código propio:
el inventario indica su situación y cómo utilizarlos; no se presentan como modelos cargados.

Los ejemplos no crean indicadores definitivos, no entrenan modelos ni procesan el corpus completo.
Las puntuaciones son salidas exploratorias, no probabilidades calibradas de creencias reales,
medidas definitivas de actos ni evidencia de causalidad económica.

## Entorno, rutas y caché

Selecciona el kernel `.venv` del proyecto. Se utilizan las dependencias existentes
`torch`, `transformers`, `huggingface-hub`, `pandas` y `truststore`, más `sentencepiece`
para tokenizadores DeBERTa/T5. Si falta esta última, desde la raíz:

```powershell
uv pip install --python .venv/Scripts/python.exe sentencepiece==0.2.3
```

Los pesos quedan en `data/models/infini_hf/`, fuera de Git. La primera descarga de los
diez modelos ocupa varios GB. `LOCAL_FILES_ONLY=True` reutiliza exclusivamente la caché;
si falta un peso, el error queda visible. No se usan APIs de inferencia ni se envían noticias
a terceros. Cada ejemplo ejecuta inferencia local. El manifiesto registra versiones,
revisión del checkpoint, etiquetas originales, dispositivo y código utilizado.

In [15]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import sys

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
import truststore
truststore.inject_into_ssl()
import pandas as pd
import torch
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/infini_models.py").is_file() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ejecuta el notebook dentro del repositorio")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.infini_models import (MODELS, PUBLIC_MODELS, EXAMPLES, DEMO_LABELS,
                              ModelBank, ModelUnavailable, write_json)

DATA_PATH = ROOT / "data/INFINI_dataset/infini_2025_eng.jsonl.gz"
MODEL_CACHE = ROOT / "data/models/infini_hf"
RUN_DIR = ROOT / "data/processed/INFINI_exploration" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
LOCAL_FILES_ONLY = True   # Pesos descargados durante la preparación; False permite descargar faltantes.
RUN_MODEL_EXAMPLES = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
torch.set_num_threads(min(4, os.cpu_count() or 1))
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
# CUDA puede conservar no determinismo; estos ejemplos no prueban reproducibilidad bit a bit.
models = ModelBank(MODEL_CACHE, RUN_DIR, device=DEVICE, local_files_only=LOCAL_FILES_ONLY)
print("Python:", sys.executable)
print("Dispositivo:", DEVICE, "| caché:", MODEL_CACHE)
print("Registro:", RUN_DIR)

c:\Users\Dccs2\Desktop\proyecto_local\DCCS-Gold-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: c:\Users\Dccs2\Desktop\proyecto_local\DCCS-Gold-Thesis\.venv\Scripts\python.exe
Dispositivo: cpu | caché: c:\Users\Dccs2\Desktop\proyecto_local\DCCS-Gold-Thesis\data\models\infini_hf
Registro: c:\Users\Dccs2\Desktop\proyecto_local\DCCS-Gold-Thesis\data\processed\INFINI_exploration\20261118T201620_898728Z


## Datos INFINI: exploración inicial

In [16]:
import pandas as pd

In [17]:
df = pd.read_json(
    DATA_PATH,
    lines=True,
    compression="gzip"
)

print(df.shape)
print(df.columns.tolist())
print(df[["title", "publish_date", "source"]].head().sort_values(by="publish_date", ascending=False))


(10000, 9)
['title', 'text', 'publish_date', 'warc_date', 'url', 'source', 'author', 'warc_record_id', 'language']
                                               title publish_date  \
2      Four Israeli hostages freed in ceasefire deal   2025-01-25   
3  Projected Lineups for the Avalanche vs Bruins ...   2025-01-25   
4  2025 NFL Draft Scouting Report: Ohio State DT ...   2025-01-25   
0  Immigrants and advocates prepare for Trump's i...   2025-01-24   
1  Why Trump's holding weekend rally in Las Vegas...   2025-01-24   

                 source  
2       www.cbsnews.com  
3  thehockeywriters.com  
4     steelersdepot.com  
0          www.nepm.org  
1       www.foxnews.com  


In [18]:
df.iloc[0].text

'Immigration court is the inevitable reality for thousands of individuals in Massachusetts. To remain here legally, going to hearing dates is a must.\nBut the court itself will soon undergo significant upheaval, from both administrative changes and caseloads that could pile on to the already strained system. That’s worrying immigrants and advocates, with some families wondering if they should avoid public spaces, including courthouses.\n“I think there are concerns about encountering ICE in almost any situation that immigrants go into who are in temporary or uncertain statuses,” said Liz Sweet, executive director of the Massachusetts Immigrant and Refugee Advocacy coalition.\nThe landmark case Ryan v. ICE blocks immigration arrests in and around Massachusetts courthouses. But that only applies to state courts, according to Middlesex County District Attorney Marian Ryan’s office. It doesn’t apply to federal immigration court.\nThere will likely be an increase of deportation cases given t

## Inventario de todas las menciones pertinentes en inglés

Se separan **checkpoint**, **familia base**, **corpus** y **sistema de investigación**.
Cuando el PDF no fija el tamaño de BERT/T5/ELECTRA se carga una variante base representativa
y se indica expresamente: no se reconstruye el ajuste del artículo a partir de pesos base.

El baseline **CNC V2** es una implementación oficial posterior para el recurso Causal News Corpus
mencionado en el PDF; no se le atribuyen las métricas del benchmark V1.
**Corrección de idioma:** FinReason es un corpus chino según su [artículo original, §1](https://aclanthology.org/2021.eacl-main.175.pdf),
aunque el PDF lo liste como inglés. Se conserva en el inventario como excluido.
También se excluyen por idioma BERTimbau/Porttinari (portugués), BETO, SELECTRA medium/small,
BETO-XNLI, DIHANA y el corpus de factualidad español. Searle e ISO 24617-2 son taxonomías, no checkpoints.

In [19]:
resources = [('RoBERTa speech-act', '1,3,10,15-16', 'checkpoint', 'Carga: speech_act', 'https://huggingface.co/quantor-project/speech-act-classification', 'Oraciones/turnos → etiquetas del checkpoint; licencia y documentación incompletas.'), ('DeBERTa xsmall zero-shot', '2,10', 'checkpoint', 'Carga: zeroshot_deberta', 'https://huggingface.co/MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33', 'Texto + hipótesis/etiquetas inglesas → scores NLI.'), ('XtremeDistil zero-shot', '2,10', 'checkpoint', 'Carga: zeroshot_xtremedistil', 'https://huggingface.co/MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33', 'Misma interfaz NLI; modelo pequeño de comparación.'), ('BERT, RoBERTa-base/large, DeBERTa-v3-base, ELECTRA', '3,7,9-10,12-15', 'familias base', 'Carga: cinco encoders, incluido RoBERTa-large', 'https://huggingface.co/docs/transformers/model_doc/auto', 'Embeddings únicamente; entrenar una cabeza exige etiquetas y un protocolo posterior.'), ('T5', '7', 'familia base', 'Carga: t5_base', 'https://huggingface.co/google-t5/t5-base', 'Texto con prefijo de tarea → generación; no es el modelo MPQA ajustado.'), ('FOMC / Trillion Dollar Words', '2,12-13', 'checkpoint', 'Acceso manual; carga local preparada', 'https://huggingface.co/gtfintechlab/FOMC-RoBERTa', 'Oración → Dovish/Hawkish/Neutral; revisión del autor y licencia CC BY-NC 4.0.'), ('Causal News Corpus', '8,15', 'corpus + baseline', 'Carga: causal_news (V2)', 'https://github.com/tanfiona/CausalNewsCorpus', 'Clasificación causal binaria. No extrae causa/efecto ni verifica causalidad real.'), ('Target Based Speech Act Classification', '1,4', 'sistema/tarea', 'Sin checkpoint directo verificado', 'https://aclanthology.org/S19-1030/', 'Texto de campaña con contexto → acto + target; requiere implementación/entrenamiento original.'), ('Epistemic Stance / ExPRES', '1,5,15', 'sistema', 'Modelos anunciados como coming soon', 'https://github.com/slanglab/ExPRES', 'Cláusula + fuente → compromiso; no sustituir por RoBERTa base sin ajustar.'), ('Modal Dependency Parsing', '1,5-6', 'corpus + parser de investigación', 'Dataset público; checkpoint no verificado', 'https://github.com/Jryao/modal_dependency', 'Tokens/eventos/conceivers → aristas child, child_type, parent, relation.'), ('BeLeaf', '1,6,15-16', 'sistema + checkpoint separado', 'Adaptador local preparado', 'https://github.com/yurpl/beleaf', 'Flan-T5 ajustado genera árboles; pipeline completo añade coreferencia y spans.'), ('BeSt', '1,6,11,15', 'corpus multilingüe con inglés', 'Requiere acceso LDC; no es un modelo', 'https://aclanthology.org/2022.lrec-1.262/', 'Anotaciones de quién cree/siente qué; entrenamiento o evaluación posterior.'), ('FactBank', '5-6,15-16', 'corpus', 'No es un checkpoint', 'https://aclanthology.org/2022.nlpcss-1.11/', 'Factualidad por evento y fuente; usar anotaciones/autorización del corpus.'), ('MPQA 2.0/3.0 y modelos BERT/T5 de private states', '1,7', 'corpus + implementaciones', 'Bases cargadas; pesos especializados no verificados', 'https://github.com/theSaeed/opinion-mining-using-llms', 'Preparar MPQA y ejecutar implementación original para tipo/polaridad/intensidad/expresión/fuente.'), ('CommitmentBank / BERT-NLI', '7', 'corpus + experimento', 'BERT base cargado; ajuste NLI específico no verificado', 'https://aclanthology.org/D19-1630/', 'Premisa + proposición embebida → compromiso/NLI; no usar BERT base como clasificador entrenado.'), ('FinCausal / GBe span model', '2,8,15', 'shared task + arquitectura propia', 'Integración original separada', 'https://github.com/guillaume-be/Financial-Causality-Extraction', 'Texto financiero → spans causa/efecto; modelo con heads propios, no QA genérico intercambiable.'), ('Causal Micro-Narratives / Llama 3.1 8B', '2,8-9,15', 'sistema + fine-tune', 'Base restringida; fine-tune exacto no verificado', 'https://github.com/heddayam/causal-micro-narratives', 'Oración + ontología y prompt del estudio → narrativas; pesos base no reproducen sus métricas.'), ('causal-narrative', '2,7-8', 'biblioteca', 'Ejemplo opcional en entorno separado', 'https://pypi.org/project/causal-narrative/', 'Patrones/clasificador/LLM, SRL y clustering; AllenNLP requiere Python 3.9–3.10, distinto de este proyecto.'), ('FinArg', '2,9', 'corpus', 'No es un checkpoint', 'https://github.com/Alaa-Ah/The-FinArg-Dataset-Argument-Mining-in-Financial-Earnings-Calls', 'Earnings calls → componentes y relaciones argumentativas; necesita modelo entrenado.'), ('FinReason', '2,9', 'corpus chino', 'Excluido: el PDF atribuye un idioma incorrecto', 'https://aclanthology.org/2021.eacl-main.175.pdf', 'No aplicar sus pesos/anotaciones como modelo inglés.'), ('DirectQuote', '2,11', 'corpus', 'No se verificó checkpoint original listo para carga', 'https://github.com/THUNLP-MT/DirectQuote', 'IOB de cita y hablante; entrenar etiquetador y conservar offsets.'), ('FactAppeal', '2,11-12', 'corpus', 'Repositorio oficial publica CSV, no checkpoint', 'https://github.com/guymorlan/factappeal', 'sentence/annotation con tags XML de factualidad y fuente; no confundir con verificación de verdad.'), ('NewsDiscourse', '2,12,15-16', 'corpus + código', 'Checkpoint original no verificado', 'https://github.com/prafulla77/Discourse_Profiling', 'Documento contextualizado → rol de cada oración. No es una clasificación independiente de oraciones.'), ('GPT / GPT-4o, Claude, Gemini', '1,3,9,16', 'APIs o familias sin versión exacta', 'No se invocan: acceso, costos y versiones pendientes', 'https://link.springer.com/article/10.1007/s41701-026-00266-0', 'Referencias/comparadores del PDF; no son pesos locales públicos ni se envían noticias a proveedores.'), ('TF-IDF + regresión logística / XGBoost', '3', 'baselines de aprendizaje', 'Sin pesos preentrenados; plantilla de entrenamiento', 'https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression', 'Necesitan etiquetas de entrenamiento; no generan actos antes del ajuste.'), ('K-Means / HDBSCAN / DP-Means, PCA y entropía', '7,15', 'métodos auxiliares', 'No son modelos de lenguaje', 'https://pypi.org/project/causal-narrative/', 'Clustering/reducción/agregación; no se fijan índices científicos en esta exploración.')]
inventory = pd.DataFrame(resources, columns=["resource", "pdf_pages", "kind", "availability", "source", "usage"])
display(inventory)
inventory.to_csv(RUN_DIR / "resource_inventory.csv", index=False, encoding="utf-8-sig")
display(pd.DataFrame([{"name": s.name, "repo_id": s.repo_id, "kind": s.kind,
                       "revision": s.revision, "restricted": s.gated} for s in MODELS.values()]))

,resource,pdf_pages,kind,availability,source,usage
0,RoBERTa speech-act,"1,3,10,15-16",checkpoint,Carga: speech_act,https://huggingface.co/quantor-project/speech-...,Oraciones/turnos → etiquetas del checkpoint; l...
1,DeBERTa xsmall zero-shot,"2,10",checkpoint,Carga: zeroshot_deberta,https://huggingface.co/MoritzLaurer/deberta-v3...,Texto + hipótesis/etiquetas inglesas → scores ...
2,XtremeDistil zero-shot,"2,10",checkpoint,Carga: zeroshot_xtremedistil,https://huggingface.co/MoritzLaurer/xtremedist...,Misma interfaz NLI; modelo pequeño de comparac...
3,"BERT, RoBERTa-base/large, DeBERTa-v3-base, ELE...","3,7,9-10,12-15",familias base,"Carga: cinco encoders, incluido RoBERTa-large",https://huggingface.co/docs/transformers/model...,Embeddings únicamente; entrenar una cabeza exi...
4,T5,7,familia base,Carga: t5_base,https://huggingface.co/google-t5/t5-base,Texto con prefijo de tarea → generación; no es...
5,FOMC / Trillion Dollar Words,"2,12-13",checkpoint,Acceso manual; carga local preparada,https://huggingface.co/gtfintechlab/FOMC-RoBERTa,Oración → Dovish/Hawkish/Neutral; revisión del...
6,Causal News Corpus,"8,15",corpus + baseline,Carga: causal_news (V2),https://github.com/tanfiona/CausalNewsCorpus,Clasificación causal binaria. No extrae causa/...
7,Target Based Speech Act Classification,"1,4",sistema/tarea,Sin checkpoint directo verificado,https://aclanthology.org/S19-1030/,Texto de campaña con contexto → acto + target;...
8,Epistemic Stance / ExPRES,"1,5,15",sistema,Modelos anunciados como coming soon,https://github.com/slanglab/ExPRES,Cláusula + fuente → compromiso; no sustituir p...
9,Modal Dependency Parsing,"1,5-6",corpus + parser de investigación,Dataset público; checkpoint no verificado,https://github.com/Jryao/modal_dependency,"Tokens/eventos/conceivers → aristas child, chi..."


,name,repo_id,kind,revision,restricted
0,speech_act,quantor-project/speech-act-classification,classification,27b8c58b65139fe7b9e1a1771062581a9d35a415,False
1,zeroshot_deberta,MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-a...,zero_shot,262ae02f29173eec1c250f90804dc7edc677dcff,False
2,zeroshot_xtremedistil,MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1....,zero_shot,c07f66d9cbf781191bee66edfe8ad7856f045781,False
3,causal_news,tanfiona/cnc-v2-st1-csc,classification,2d036aaff58d971bcc2599465a956d512294c90a,False
4,roberta_base,FacebookAI/roberta-base,encoder,e2da8e2f811d1448a5b465c236feacd80ffbac7b,False
5,roberta_large,FacebookAI/roberta-large,encoder,722cf37b1afa9454edce342e7895e588b6ff1d59,False
6,deberta_base,microsoft/deberta-v3-base,encoder,8ccc9b6f36199bec6961081d44eb72fb3f7353f3,False
7,bert_base,google-bert/bert-base-uncased,encoder,86b5e0934494bd15c9632b12f734a8a67f723594,False
8,electra_base,google/electra-base-discriminator,encoder,1ae76a97c7e84a4e640876a07453fccd636f0667,False
9,t5_base,google-t5/t5-base,seq2seq,a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1,False


## Función para ejecutar ejemplos y registrar fallos

Cada celda carga el modelo que necesita y libera el anterior. Todos los pesos permanecen en
disco. No se mantiene una copia de los diez modelos en RAM/VRAM. `example_reports` distingue
ejemplos ejecutados, desactivados y fallidos; un fallo no se convierte en una predicción vacía.

In [20]:
example_reports = []

def run_example(name, operation):
    if not RUN_MODEL_EXAMPLES:
        print("Ejemplo desactivado:", name)
        return None
    try:
        result = operation()
        display(result)
        report = {"name": name, "status": "passed"}
    except Exception as exc:
        result = None
        report = {"name": name, "status": "failed", "error_type": type(exc).__name__, "error": str(exc)[:1200]}
        display(report)
    example_reports.append(report)
    write_json(RUN_DIR / "example_reports.json", example_reports)
    return result

## 1. RoBERTa: actos de habla

[Checkpoint citado en el PDF](https://huggingface.co/quantor-project/speech-act-classification).
**Entrada:** turno u oración en inglés. **Salida:** score por etiqueta del checkpoint.
La población de entrenamiento descrita en el PDF es conversación oral: transferir a noticias
requiere evaluación. La tarjeta pública está incompleta. Se muestra `id2label` sin imponer
el orden de Searle: si solo hay `LABEL_0`, etc., no se adivina qué significa cada número.

In [21]:
speech_result = run_example("speech_act", lambda: models.classify("speech_act", EXAMPLES))
if models.active_name == "speech_act":
    print("Etiquetas originales:", models.model.config.id2label)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 684.41it/s]


[{'text': 'The central bank expects inflation to decline next year.',
  'scores': [{'label': '<com>',
    'label_description': None,
    'score': 0.0010525446850806475},
   {'label': '<dec>',
    'label_description': None,
    'score': 0.000404799182433635},
   {'label': '<dir>',
    'label_description': None,
    'score': 0.014878939837217331},
   {'label': '<exp>',
    'label_description': None,
    'score': 0.0010468516265973449},
   {'label': '<icu>',
    'label_description': None,
    'score': 0.0011080893455073237},
   {'label': '<rep>', 'label_description': None, 'score': 0.9800028800964355},
   {'label': '<soc>',
    'label_description': None,
    'score': 0.00027758919168263674},
   {'label': '<xpa>',
    'label_description': None,
    'score': 0.0012283463729545474}]},
 {'text': 'Higher borrowing costs caused investment to fall.',
  'scores': [{'label': '<com>',
    'label_description': None,
    'score': 0.0003027870843652636},
   {'label': '<dec>',
    'label_description': 

Etiquetas originales: {0: '<com>', 1: '<dec>', 2: '<dir>', 3: '<exp>', 4: '<icu>', 5: '<rep>', 6: '<soc>', 7: '<xpa>'}


## DeBERTa-v3-xsmall zero-shot

[Ficha del autor](https://huggingface.co/MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33).
**Entrada:** texto inglés y etiquetas escritas en inglés. **Salida:** score de entailment
frente a contradicción por etiqueta. Se evalúan independientemente; los scores no tienen
por qué sumar uno entre categorías. `DEMO_LABELS` es solo una demostración, no una taxonomía
aprobada. Cambiar la plantilla o las etiquetas cambia el experimento y debe registrarse.

In [22]:
zeroshot_deberta_result = run_example("zeroshot_deberta", lambda: pd.DataFrame(
    models.zero_shot("zeroshot_deberta", EXAMPLES[0], DEMO_LABELS)))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 3995.59it/s]


,label,score,hypothesis
0,reports a fact,0.166748,This sentence reports a fact.
1,makes a forecast,0.948242,This sentence makes a forecast.
2,expresses a warning,0.658691,This sentence expresses a warning.
3,makes a commitment,0.870605,This sentence makes a commitment.


## XtremeDistil zero-shot

[Ficha del autor](https://huggingface.co/MoritzLaurer/xtremedistil-l6-h256-zeroshot-v1.1-all-33).
**Entrada:** texto inglés y etiquetas escritas en inglés. **Salida:** score de entailment
frente a contradicción por etiqueta. Se evalúan independientemente; los scores no tienen
por qué sumar uno entre categorías. `DEMO_LABELS` es solo una demostración, no una taxonomía
aprobada. Cambiar la plantilla o las etiquetas cambia el experimento y debe registrarse.

In [23]:
zeroshot_xtremedistil_result = run_example("zeroshot_xtremedistil", lambda: pd.DataFrame(
    models.zero_shot("zeroshot_xtremedistil", EXAMPLES[0], DEMO_LABELS)))

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 26250.34it/s]


,label,score,hypothesis
0,reports a fact,0.157471,This sentence reports a fact.
1,makes a forecast,0.973633,This sentence makes a forecast.
2,expresses a warning,0.475098,This sentence expresses a warning.
3,makes a commitment,0.234009,This sentence makes a commitment.


## Causal News Corpus: detección binaria

El [baseline oficial CNC V2](https://huggingface.co/tanfiona/cnc-v2-st1-csc) distingue
`LABEL_0 = Non-causal` y `LABEL_1 = Causal`. Se conserva tanto el identificador como la
descripción documentada. Detecta causalidad expresada en el texto; no identifica efectos
económicos. No devuelve spans de causa/efecto. **FinCausal es otro recurso**, no un alias
de este checkpoint. La ficha del baseline declara licencia desconocida.

Compatibilidad con Transformers 5: el archivo original usa enteros en `id2label`.
El cargador los convierte a strings en memoria y registra esa conversión; no modifica
el archivo descargado, el orden de las clases ni los pesos. Por eso la salida puede
mostrar `0`/`1`, además de la descripción documentada Non-causal/Causal.

In [24]:
causal_result = run_example("causal_news", lambda: models.classify("causal_news", [
    "Higher borrowing costs caused investment to fall.",
    "The company published its annual report on Tuesday.",
]))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28710.50it/s]


[{'text': 'Higher borrowing costs caused investment to fall.',
  'scores': [{'label': '0',
    'label_description': 'Non-causal',
    'score': 0.0009230658761225641},
   {'label': '1',
    'label_description': 'Causal',
    'score': 0.9990769624710083}]},
 {'text': 'The company published its annual report on Tuesday.',
  'scores': [{'label': '0',
    'label_description': 'Non-causal',
    'score': 0.9991706609725952},
   {'label': '1',
    'label_description': 'Causal',
    'score': 0.0008293244172818959}]}]

## BERT-base-uncased: familia nombrada en los experimentos MPQA y CommitmentBank

**Entrada:** texto inglés. **Salida de este ejemplo:** vector obtenido mediante promedio
de los estados de tokens usando la máscara de atención. Este pooling es una elección técnica
ilustrativa, no un indicador aprobado ni un modelo ajustado de sentimiento/creencias.
El encoder no contiene una cabeza entrenada para esas tareas.

In [25]:
def demo_bert_base():
    vectors = models.embed("bert_base", EXAMPLES[:2])
    return {"shape": tuple(vectors.shape), "finite": bool(torch.isfinite(vectors).all()),
            "first_dimensions": vectors[0, :8].tolist()}

bert_base_result = run_example("bert_base", demo_bert_base)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1112.43it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'shape': (2, 768),
 'finite': True,
 'first_dimensions': [-0.13227231800556183,
  -0.30794087052345276,
  0.42269089818000793,
  0.0826500952243805,
  0.3598545789718628,
  -0.027624113485217094,
  -0.14385397732257843,
  0.8402320742607117]}

## RoBERTa-base: encoder propuesto para adaptar al dominio

**Entrada:** texto inglés. **Salida de este ejemplo:** vector obtenido mediante promedio
de los estados de tokens usando la máscara de atención. Este pooling es una elección técnica
ilustrativa, no un indicador aprobado ni un modelo ajustado de sentimiento/creencias.
El encoder no contiene una cabeza entrenada para esas tareas.

In [ ]:
def demo_roberta_base():
    vectors = models.embed("roberta_base", EXAMPLES[:2])
    return {"shape": tuple(vectors.shape), "finite": bool(torch.isfinite(vectors).all()),
            "first_dimensions": vectors[0, :8].tolist()}

roberta_base_result = run_example("roberta_base", demo_roberta_base)

## RoBERTa-large: arquitectura base mencionada para FOMC, sin su ajuste especializado

**Entrada:** texto inglés. **Salida de este ejemplo:** vector obtenido mediante promedio
de los estados de tokens usando la máscara de atención. Este pooling es una elección técnica
ilustrativa, no un indicador aprobado ni un modelo ajustado de sentimiento/creencias.
El encoder no contiene una cabeza entrenada para esas tareas.

In [ ]:
def demo_roberta_large():
    vectors = models.embed("roberta_large", EXAMPLES[:2])
    return {"shape": tuple(vectors.shape), "finite": bool(torch.isfinite(vectors).all()),
            "first_dimensions": vectors[0, :8].tolist()}

roberta_large_result = run_example("roberta_large", demo_roberta_large)

## DeBERTa-v3-base: encoder alternativo para fine-tuning

**Entrada:** texto inglés. **Salida de este ejemplo:** vector obtenido mediante promedio
de los estados de tokens usando la máscara de atención. Este pooling es una elección técnica
ilustrativa, no un indicador aprobado ni un modelo ajustado de sentimiento/creencias.
El encoder no contiene una cabeza entrenada para esas tareas.

In [ ]:
def demo_deberta_base():
    vectors = models.embed("deberta_base", EXAMPLES[:2])
    return {"shape": tuple(vectors.shape), "finite": bool(torch.isfinite(vectors).all()),
            "first_dimensions": vectors[0, :8].tolist()}

deberta_base_result = run_example("deberta_base", demo_deberta_base)

## ELECTRA-base-discriminator en inglés; distinto de SELECTRA español

**Entrada:** texto inglés. **Salida de este ejemplo:** vector obtenido mediante promedio
de los estados de tokens usando la máscara de atención. Este pooling es una elección técnica
ilustrativa, no un indicador aprobado ni un modelo ajustado de sentimiento/creencias.
El encoder no contiene una cabeza entrenada para esas tareas.

In [ ]:
def demo_electra_base():
    vectors = models.embed("electra_base", EXAMPLES[:2])
    return {"shape": tuple(vectors.shape), "finite": bool(torch.isfinite(vectors).all()),
            "first_dimensions": vectors[0, :8].tolist()}

electra_base_result = run_example("electra_base", demo_electra_base)

## T5: generación condicionada por una tarea

[T5-base](https://huggingface.co/google-t5/t5-base) recibe texto con prefijo de tarea.
El ejemplo usa `summarize:` y decodificación determinista sobre texto sintético. El resumen
puede omitir o inventar detalles: no reemplaza el cuerpo original ni demuestra capacidad
de extraer private states. Los modelos BERT/T5 ajustados sobre MPQA del artículo requieren
sus propios pesos, formatos y etiquetas.

In [ ]:
t5_result = run_example("t5_base", lambda: models.generate("t5_base", [
    "summarize: The central bank kept interest rates unchanged. Officials said inflation remained too high."
], max_new_tokens=40))

## FOMC-RoBERTa y Llama 3.1 8B: cargas preparadas para pesos autorizados

[FOMC-RoBERTa](https://huggingface.co/gtfintechlab/FOMC-RoBERTa) exige aprobación manual.
Se utiliza el orden documentado `0=Dovish, 1=Hawkish, 2=Neutral`; no se añade una clase N/A
que el checkpoint no tiene. El usuario puede proporcionar un directorio local completo con
configuración, pesos y tokenizador. No se leen ni se solicitan credenciales desde el notebook.

Llama 3.1 8B también requiere acceso; el modelo base no es el fine-tune de
[Causal Micro-Narratives](https://github.com/heddayam/causal-micro-narratives).
El directorio local debe indicar qué variante contiene. Cargar 8B necesita bastante RAM/VRAM;
en este equipo se detectó CPU durante la preparación. No se descarga automáticamente.

In [ ]:
FOMC_LOCAL_DIR = None       # Ej.: ROOT / "data/models/FOMC-RoBERTa"
LLAMA31_LOCAL_DIR = None    # Base local autorizada; no el fine-tune de Causal Micro-Narratives.

if FOMC_LOCAL_DIR is not None:
    models.local_overrides["fomc"] = FOMC_LOCAL_DIR
    fomc_result = run_example("fomc", lambda: models.classify("fomc", EXAMPLES))
else:
    print("FOMC: pendiente de acceso manual/checkpoint local autorizado.")

if LLAMA31_LOCAL_DIR is not None:
    models.local_overrides["llama31_8b"] = LLAMA31_LOCAL_DIR
    llama_result = run_example("llama31_8b", lambda: models.generate("llama31_8b", [
        "The central bank said inflation was likely to"
    ], max_new_tokens=32))
else:
    print("Llama 3.1 8B: pendiente de acceso y pesos locales; no se equipara al ajuste del paper.")

## BeLeaf: ejemplo del generador de árboles con su checkpoint ajustado

El [README oficial](https://github.com/yurpl/beleaf) enlaza el checkpoint en
[esta carpeta de los autores](https://drive.google.com/drive/folders/1PwwxxfCw_UPcbsbFa3-QSyuzK0xSY5MF).
El [código de inferencia](https://github.com/yurpl/beleaf/blob/main/api/inference.py)
usa los pesos ajustados y el tokenizador `google/flan-t5-large`.

La celda siguiente muestra la **generación cruda del árbol** a partir de una oración.
No ejecuta la resolución de coreferencias ni la conversión de heads a spans del sistema
completo; para ello se necesita su entorno original (`spaCy` y `fastcoref`). No se cargan
pesos Flan-T5 base fingiendo que son el ajuste BeLeaf. Si no se dispone del checkpoint,
se muestra el estado pendiente y no se genera una estructura ficticia.

In [ ]:
BELEAF_LOCAL_DIR = None  # Directorio del checkpoint ajustado source_target_ckpt de los autores.

if BELEAF_LOCAL_DIR is not None and RUN_MODEL_EXAMPLES:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    models.release()
    beleaf_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large", cache_dir=MODEL_CACHE,
        local_files_only=LOCAL_FILES_ONLY, token=False, trust_remote_code=False)
    beleaf_model = AutoModelForSeq2SeqLM.from_pretrained(str(BELEAF_LOCAL_DIR),
        local_files_only=True, trust_remote_code=False).to(DEVICE).eval()
    inputs = beleaf_tokenizer(["Analysts believe the Fed expects inflation to decline."],
                             return_tensors="pt", padding=True, truncation=False).to(DEVICE)
    with torch.inference_mode():
        generated = beleaf_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    print("Árbol generado, sin postprocesamiento:", beleaf_tokenizer.batch_decode(generated, skip_special_tokens=True))
    del beleaf_model, beleaf_tokenizer, inputs, generated
    gc.collect()
else:
    print("BeLeaf: pendiente del checkpoint ajustado externo; véase enlace oficial.")

## Cómo trabajar con los restantes recursos

| Recurso | Flujo que permite usarlo |
|---|---|
| Epistemic Stance / ExPRES | El repo aún dice `coming soon`. Obtener pesos y esquema de fuentes/cláusulas del autor antes de inferir; RoBERTa-base no es sustituto. |
| Modal Dependency Parsing | El corpus define aristas `child, child_type, parent, relation` sobre offsets de tokens. Se necesita el parser ajustado y su preprocesamiento; no es clasificación de oraciones. |
| BeSt, FactBank, MPQA | Obtener corpus y permisos aplicables, inspeccionar anotaciones de fuente/target/evento y entrenar o evaluar una implementación; no hay una llamada genérica `pipeline(corpus)`. |
| CommitmentBank | Leer premisa e hipótesis y usar un modelo NLI ajustado. Los dos zero-shot cargados permiten demostrar NLI, pero no son el BERT del artículo. |
| Target Based Speech Act | Reproducir modelo secuencial acto-target con contexto y etiquetas del paper; sin checkpoint verificado, no se inventa un identificador HF. |
| FinCausal / GBe | Usar las clases de span y la evaluación del [repo original](https://github.com/guillaume-be/Financial-Causality-Extraction). Una cabeza QA genérica no tiene sus salidas. |
| Causal Micro-Narratives | Recuperar fine-tune, ontología y prompts de los autores. Cargar Llama base solo verifica la arquitectura y no replica ese modelo. |
| FinArg | Leer las anotaciones de componentes y relaciones; después entrenar un clasificador/etiquetador con su esquema. |
| DirectQuote | Leer CoNLL IOB1 y conservar offsets/relación con hablante; entrenar etiquetador antes de inferir atribuciones. |
| FactAppeal | El [repositorio oficial](https://github.com/guymorlan/factappeal) publica `sentence` y `annotation` en CSV con tags XML; preparar etiquetas y ajustar un modelo. |
| NewsDiscourse | Ejecutar implementación del autor con documentos completos y sus roles; conserva contexto entre oraciones. Pesos de terceros, si se usan, deben registrarse como otro modelo. |

Estas integraciones quedan explícitas como pendientes cuando faltan pesos, acceso o un entorno
compatible. No se fabrican resultados ni se entrenan cabezas nuevas sin anotaciones.

`causal-narrative` es una biblioteca, no un checkpoint único. Su [documentación](https://pypi.org/project/causal-narrative/)
incluye el flujo SRL inglés `from causal_narrative import get_srl; srl = get_srl('spacy');
srl.process(text)`. El backend spaCy necesita `en_core_web_sm`; AllenNLP necesita un entorno
Python 3.9–3.10 y no es compatible con el requisito Python 3.11 de este proyecto.
No se modifica el entorno del proyecto para resolver conflictos de esa biblioteca.

In [ ]:
# Lectura local de anotaciones: no descarga corpus ni entrena un modelo.
FACTAPPEAL_LOCAL_CSV = None
if FACTAPPEAL_LOCAL_CSV is not None:
    annotations = pd.read_csv(FACTAPPEAL_LOCAL_CSV)
    required_columns = {"sentence", "annotation"}
    if not required_columns.issubset(annotations.columns):
        raise ValueError("Se esperaba el formato sentence/annotation del repositorio oficial")
    display(annotations[["sentence", "annotation"]].head())

# Baseline del PDF: se instancia, pero requiere un corpus anotado antes de .fit().
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
speech_act_baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)),
                                    LogisticRegression(max_iter=1000, random_state=42))
print("Baseline TF-IDF/logística preparado, sin entrenar; no produce predicciones todavía.")
# XGBoost también requiere etiquetas; no se presenta como un modelo lingüístico preentrenado.

## Ejemplo acotado sobre tus noticias INFINI

Se seleccionan las dos primeras filas con texto únicamente como demostración de ingeniería.
No es una muestra representativa ni un criterio de inclusión científico. Para evitar truncar
el artículo, se construyen ventanas técnicas con offsets. El ejemplo procesa **solo la primera
ventana de cada noticia**, deja explícito ese alcance y conserva texto/hash/URL/fechas.
Las otras ventanas permanecen disponibles para un experimento posterior.

Los solapamientos hacen que una agregación posterior necesite reglas para no contar dos veces
el mismo fragmento. No se agregan automáticamente scores por artículo ni se calcula un índice.

In [ ]:
RUN_INFINI_EXAMPLE = True
MAX_NEWS_EXAMPLES = 2

def demo_infini():
    rows = []
    selected = df.loc[df["text"].map(lambda value: isinstance(value, str) and bool(value.strip()))].head(MAX_NEWS_EXAMPLES)
    for row_index, row in selected.iterrows():
        segments = models.windows("zeroshot_xtremedistil", row["text"], max_tokens=192, overlap=24)
        if not segments:
            continue
        segment = segments[0]
        predictions = models.zero_shot("zeroshot_xtremedistil", segment["text"], DEMO_LABELS)
        rows.append({"row_index": str(row_index), "url": row.get("url"), "title": row.get("title"),
            "publish_date": str(row.get("publish_date")), "warc_date": str(row.get("warc_date")),
            "source": row.get("source"), "article_text_sha256": hashlib.sha256(row["text"].encode()).hexdigest(),
            **segment, "total_windows": len(segments), "windows_processed": 1,
            "model_id": MODELS["zeroshot_xtremedistil"].repo_id,
            "model_revision": MODELS["zeroshot_xtremedistil"].revision,
            "hypothesis_template": "This sentence {}.", "candidate_labels": DEMO_LABELS,
            "scores": predictions, "scope": "first window only; exploratory"})
    write_json(RUN_DIR / "infini_example.json", rows)
    return pd.DataFrame([{k: r[k] for k in ["row_index", "source", "total_windows", "windows_processed", "scores"]} for r in rows])

if RUN_INFINI_EXAMPLE:
    infini_result = run_example("INFINI_first_windows", demo_infini)

## Estado y trazabilidad

Un `passed` confirma carga e inferencia técnica del ejemplo, no validez científica en INFINI.
`models_manifest.json` registra cargas y revisiones; `example_reports.json` registra qué ejemplos
se ejecutaron. El inventario completo se guarda en `resource_inventory.csv`. El original INFINI
permanece en solo lectura; `infini_example.json` contiene únicamente los fragmentos utilizados.

Para repetir los ejemplos, conserva las revisiones fijas y registra cualquier cambio de etiquetas,
plantilla, segmentación, dispositivo y preprocesamiento. El PDF contiene propuestas de índices y
taxonomías que este notebook no adopta como decisiones científicas definitivas.

In [ ]:
display(pd.DataFrame(example_reports))
display(pd.DataFrame([{"name": name, **status} for name, status in models.status.items()]))
write_json(RUN_DIR / "run_settings.json", {"seed": 42, "device": DEVICE, "threads": torch.get_num_threads(),
    "demo_labels": DEMO_LABELS, "hypothesis_template": "This sentence {}.",
    "data_path": str(DATA_PATH), "data_size_bytes": DATA_PATH.stat().st_size,
    "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "notebook_sha256": hashlib.sha256((ROOT / "notebooks/exploracion_INFINI.ipynb").read_bytes()).hexdigest(),
    "processed_entire_corpus": False, "max_news_examples": MAX_NEWS_EXAMPLES,
    "scope": "technical examples; no scientific indicators"})
models.release()
print("Resultados y manifiestos:", RUN_DIR)

### Verificación realizada al preparar estas celdas

El 2026-11-18 se descargaron los diez checkpoints públicos en la caché local y se ejecutaron
las **21 celdas de código**: diez ejemplos de modelos y la demostración sobre las primeras
ventanas de dos noticias INFINI terminaron correctamente, en CPU y sin solicitudes de red
durante la inferencia. También pasaron **8 pruebas unitarias** de etiquetas NLI, rechazo de
cabezas aleatorias, control de longitud, offsets y acceso restringido.

Evidencia local: `data/diagnostics/infini_models/validation.json` y
`data/diagnostics/infini_models/notebook_execution.log`. Los manifiestos de esa ejecución
están en la ruta `run_dir` indicada por `validation.json`. Los ejemplos no validan exactitud
lingüística sobre INFINI. FOMC, Llama y BeLeaf no se cargaron: las celdas documentan el
acceso/checkpoint externo que falta. Los corpus no se cuentan como modelos ejecutados.